# 図形検出 (終了はescキー)


#### cv2.HoughCircles は、画像から円を検出するための関数で、ハフ変換（Hough Circle Transform）を用いて円の中心と半径を求める。

cv2.HoughCircles(
    image,
    method,
    dp,
    minDist,
    param1,
    param2,
    minRadius,
    maxRadius
)

- dp → 精度と速度のバランス,高精度（dp = 1）
- minDist → 重複検出の制御, 円同士は十分に離れている必要あり（minDist 大きめ）
- param1 → エッジの強さ, 強いエッジのみ使用（param1 = 200）
- param2（最重要） → 円の検出厳しさ, 判定はやや緩め（param2 = 20）
- min/maxRadius → 検出サイズの制限, 中サイズの円を対象（半径15〜60）

#####  パラメータ調整のコツ:
🔻 円が検出されない場合
param2 を下げる（例：20 → 15）
param1 を下げる（例：200 → 100）

🔻 誤検出が多い場合
param2 を上げる（例：20 → 30）
minDist を大きくする

🔻 小さい円を検出したい
minRadius を小さくする

🔻 大きい円を検出したい
maxRadius を大きくする

In [1]:
import cv2
import numpy as np

#画像からエッジを検出する
def canny(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    canny = cv2.Canny(blur, 50, 150)
    return canny

cap = cv2.VideoCapture(0)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

while True:
    ret, frame = cap.read()
    if ret:
        #画像の左右を自分から見た場合と同じようにする 
        frame= cv2.flip(frame, 1)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_blur = cv2.medianBlur(gray, 5)
        canny_frame = canny(frame_blur)
        
        circles = cv2.HoughCircles(canny_frame, cv2.HOUGH_GRADIENT, 1, frame.shape[0] / 4, param1=200, param2=20, minRadius=15, maxRadius=60)
        
        if circles is not None:  # If circles are detected
            circles = np.uint16(np.around(circles))  # Round the floating-point values and store them in the same variable

            for i in circles[0, :]:  # Iterate through detected circles
                # Extract circle center (i[0], i[1]) and radius (i[2])
                cv2.circle(frame, (i[0], i[1]), i[2], (255, 0, 0), 4)  # Draw the detected circles on the image
        
        # Display the image with detected circles
        cv2.imshow("frame", frame)
        
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()